# 01 — Data Inspection

**Project:** Enron E-Comms Surveillance NLP  
**Purpose:** Inspect the raw and parsed Enron email data before deeper NLP, risk scoring, and network analysis.

This notebook is intentionally lightweight. The production pipeline lives in `scripts/`; this notebook is for explanation, validation, and portfolio readability.


## What this notebook checks

- Raw dataset location and file size
- Parsed email table shape and schema
- Parse error rate
- Field coverage
- Body length distribution
- Date parsing quality
- Basic sender/recipient activity
- Dashboard-ready KPI outputs

The goal is not to rebuild the ETL pipeline here, but to show that the pipeline outputs are sensible.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

# Notebook path: notebooks/01_data_inspection.ipynb
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

RAW_FILE = PROJECT_ROOT / "data" / "raw" / "emails.csv"
PARSED_FILE = PROJECT_ROOT / "data" / "interim" / "parsed_emails.parquet"
RISK_FILE = PROJECT_ROOT / "data" / "processed" / "email_risk_scores.parquet"
DASH_DIR = PROJECT_ROOT / "data" / "dashboard"

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

print("Project root:", PROJECT_ROOT)

## 1. Raw file check

The raw Kaggle-style Enron dataset is a large flat CSV with two main fields: `file` and `message`. The `message` field contains RFC-style email text with headers and body.


In [ ]:
if RAW_FILE.exists():
    size_mb = RAW_FILE.stat().st_size / (1024 ** 2)
    size_gb = RAW_FILE.stat().st_size / (1024 ** 3)
    print(f"Raw file exists: {RAW_FILE}")
    print(f"Size: {size_mb:,.2f} MB / {size_gb:,.2f} GB")
else:
    print("Raw file not found. This is expected if raw data is excluded from Git.")

## 2. Load parsed email data

The heavy parsing step is performed by `scripts/01_parse_emails.py`. It converts raw email text into structured fields such as sender, recipient, subject, date, folder, and body.


In [ ]:
parsed = pd.read_parquet(PARSED_FILE)
print("Parsed shape:", parsed.shape)
parsed.head(3)

In [ ]:
schema = pd.DataFrame({
    "column": parsed.columns,
    "dtype": [str(dtype) for dtype in parsed.dtypes],
    "non_null": [parsed[col].notna().sum() for col in parsed.columns],
})
schema["coverage_pct"] = (schema["non_null"] / len(parsed) * 100).round(2)
schema

## 3. Parse quality

For this project, a tiny number of parsing failures is acceptable because the dataset contains over half a million historical emails. The failures are retained in `parse_error` for transparency.


In [ ]:
parse_errors = parsed[parsed["parse_error"].notna()].copy()
parse_error_count = len(parse_errors)
parse_error_rate = parse_error_count / len(parsed) * 100

print(f"Parse errors: {parse_error_count:,}")
print(f"Parse error rate: {parse_error_rate:.6f}%")

parse_errors[["file", "parse_error"]].head(10)

## 4. Field coverage

Coverage matters because downstream risk scoring and network construction depend on reliable sender, recipient, subject, body, and date fields.


In [ ]:
coverage_cols = [
    "message_id", "date", "from_email", "to_email", "cc_email",
    "bcc_email", "subject", "body", "x_folder", "x_origin"
]

coverage = []
for col in coverage_cols:
    if col in parsed.columns:
        populated = parsed[col].notna().sum()
        coverage.append({
            "field": col,
            "populated": populated,
            "missing": len(parsed) - populated,
            "coverage_pct": round(populated / len(parsed) * 100, 2),
        })

pd.DataFrame(coverage).sort_values("coverage_pct", ascending=False)

## 5. Body length profile

Email body length helps identify very short messages, long forwarded chains, and possible outliers.


In [ ]:
body_length = parsed["body"].fillna("").astype(str).str.len()
body_summary = body_length.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).to_frame("body_length")
body_summary

In [ ]:
longest = parsed.assign(body_length=body_length).sort_values("body_length", ascending=False)
longest[["file", "from_email", "to_email", "subject", "body_length"]].head(10)

## 6. Date quality

Date normalization is handled by `scripts/09_normalize_dates.py`. Some extreme date outliers may exist in historical email data, so dashboard views should default to a sensible business period such as 1998–2002.


In [ ]:
risk = pd.read_parquet(RISK_FILE, columns=["date", "date_parsed", "email_year", "email_month_period", "final_risk_score", "risk_band"])

parsed_dates = risk["date_parsed"].notna().sum()
print(f"Parsed dates: {parsed_dates:,} / {len(risk):,} ({parsed_dates / len(risk) * 100:.2f}%)")
print("Min date:", risk["date_parsed"].min())
print("Max date:", risk["date_parsed"].max())

risk[["date", "date_parsed", "email_year", "email_month_period"]].head()

In [ ]:
year_counts = (
    risk.dropna(subset=["email_year"])
        .assign(email_year=lambda d: d["email_year"].astype(int))
        .groupby("email_year", as_index=False)
        .size()
        .rename(columns={"size": "email_count"})
        .sort_values("email_year")
)

year_counts.tail(15)

## 7. Sender and recipient activity

The first communication-level view is simple activity: who sends the most, and which recipient fields are most frequent. The true network layer is handled later in the pipeline.


In [ ]:
top_senders = (
    parsed["from_email"]
    .value_counts()
    .head(15)
    .rename_axis("sender")
    .reset_index(name="email_count")
)

top_senders

In [ ]:
top_recipient_strings = (
    parsed["to_email"]
    .value_counts()
    .head(15)
    .rename_axis("recipient_field")
    .reset_index(name="email_count")
)

top_recipient_strings

## 8. Dashboard KPI outputs

The pipeline exports lightweight dashboard-ready data. This keeps Streamlit fast and avoids repeatedly processing large raw files.


In [ ]:
kpi_file = DASH_DIR / "kpi_summary.csv"
risk_band_file = DASH_DIR / "risk_band_summary.csv"
monthly_file = DASH_DIR / "monthly_email_volume.csv"

kpis = pd.read_csv(kpi_file)
risk_bands = pd.read_csv(risk_band_file)
monthly = pd.read_csv(monthly_file)

display(kpis)
display(risk_bands)
display(monthly.head(12))

## 9. Key observations

- The parsed dataset contains over **517k emails**.
- Parsing quality is very high, with only a small number of retained parse errors.
- Sender and recipient coverage is strong enough for communication network construction.
- Body lengths are highly skewed, which is expected for email data because of forwarded chains and long copied threads.
- Date parsing is mostly successful, but timeline views should use sensible business-period filters to avoid historical outliers.
- The dataset is suitable for downstream text features, risk phrase detection, network analytics, and dashboarding.


## Next notebook

`02_email_parsing_eda.ipynb` should focus on the parsed email structure, examples of headers/body extraction, and practical parsing caveats.
